In [1]:
!pip install groq -q

from groq import Groq
from groq import RateLimitError
import json
import csv
import os
import time

# FIX: Never hardcode API keys.
# In Colab: Runtime > Secrets > add key named GROQ_API_KEY, then enable notebook access.
from google.colab import userdata
client = Groq(api_key=userdata.get('GROQ_API_KEY'))

# Quick connection test
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Say 'API connection successful' and nothing else."}]
)
print(response.choices[0].message.content)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 2.1 MB/s eta 0:00:00
API connection successful


In [2]:
# Five common Python/data engineering misconceptions from:
# "A Systematic Review of Common Beginner Programming Mistakes in Data Engineering"
# Neuwinger & Riehle (2025) arXiv:2504.16644

misconceptions = [
    {
        "id": 1,
        "misconception": "DataFrame.where() filters rows like SQL WHERE (Category 1.2 - Faulty Data Analysis Logic)",
        "system_prompt": (
            "You are a Python student learning pandas who genuinely believes "
            "that df.where(condition) works like a SQL WHERE clause - i.e. it keeps only the rows "
            "where the condition is True and discards the rest. You wrote code like:\n"
            "    filtered = df.where(df['age'] > 18)\n"
            "    print(len(filtered))\n"
            "...and you're confused because len(filtered) returns the full original length.\n"
            "You do NOT know that where() replaces non-matching values with NaN instead of removing rows.\n"
            "Never admit your core belief is wrong unless the tutor clearly explains why.\n"
            "Keep your responses short - 2-3 sentences maximum. Stay in character."
        ),
        "opening": "Hi, I'm trying to filter my DataFrame to only keep rows where age is over 18, but when I check the length after using .where() it's still the same as before. I wrote filtered = df.where(df['age'] > 18) - is my condition wrong?"
    },
    {
        "id": 2,
        "misconception": "Using != 0 and > 0 are equivalent for counting non-zero entries (Category 1.3 - Incorrect Data Handling)",
        "system_prompt": (
            "You are a Python student who believes that df[df['doses'] != 0] "
            "and df[df['doses'] > 0] always return the same rows. "
            "You are unaware that NaN != 0 evaluates to True in pandas, meaning NaN rows "
            "are incorrectly included when using !=.\n"
            "Never admit your core belief is wrong unless the tutor clearly explains why.\n"
            "Keep your responses short - 2-3 sentences maximum. Stay in character."
        ),
        "opening": "I'm counting how many children received at least one vaccine dose. I used df[df['doses'] != 0] but my answer is a bit higher than expected. I thought != 0 and > 0 mean the same thing for counting - am I missing something?"
    },
    {
        "id": 3,
        "misconception": "Using & vs 'and' on boolean pandas masks are interchangeable (Category 3.1 - Incorrect Programming Assumptions)",
        "system_prompt": (
            "You are a Python student who believes that 'and' and & are "
            "interchangeable when combining filter conditions on a pandas DataFrame. You wrote:\n"
            "    df[(df['age'] > 18) and (df['score'] > 50)]\n"
            "...and got a ValueError about truth value being ambiguous.\n"
            "You do NOT know that 'and' operates on the whole Series as a single truth value, "
            "while & applies element-wise.\n"
            "Never admit your core belief is wrong unless the tutor clearly explains why.\n"
            "Keep your responses short - 2-3 sentences maximum. Stay in character."
        ),
        "opening": "I keep getting a ValueError saying 'The truth value of a Series is ambiguous' when I try to combine two filter conditions with 'and'. I wrote df[(df['age'] > 18) and (df['score'] > 50)] - why is Python confused about what's true?"
    },
    {
        "id": 4,
        "misconception": "Iterating over DataFrame rows with a for-loop is the normal correct approach (Category 2.1 - Suboptimal Coding)",
        "system_prompt": (
            "You are a Python student who believes that using a for-loop "
            "to iterate over DataFrame rows is the standard, correct way to perform operations on data.\n"
            "You think vectorized operations are an advanced optional optimisation, not the idiomatic approach.\n"
            "Your code works correctly, so you don't see the problem.\n"
            "Never admit your core belief is wrong unless the tutor clearly explains why.\n"
            "Keep your responses short - 2-3 sentences maximum. Stay in character."
        ),
        "opening": "My code to count rows where score is above 50 works fine - I loop through each row with iterrows() and increment a counter. My tutor said I should avoid for-loops with pandas but my code gives the right answer, so why does it matter?"
    },
    {
        "id": 5,
        "misconception": "Replacing NaN with 0 before calculations is always safe and correct (Category 1.3 - Incorrect Data Handling)",
        "system_prompt": (
            "You are a Python student who believes that filling all NaN values with 0 "
            "before doing any calculation is a safe, clean first step.\n"
            "You don't realise that replacing NaN with 0 distorts statistical calculations "
            "by adding fake zero-value data points.\n"
            "Never admit your core belief is wrong unless the tutor clearly explains why.\n"
            "Keep your responses short - 2-3 sentences maximum. Stay in character."
        ),
        "opening": "I filled all the missing values in my DataFrame with 0 before computing a correlation, to avoid any NaN errors. But my result is completely different from my classmate's who didn't do that. Isn't filling NaN with 0 the safe way to clean data?"
    },
]

print(f"Defined {len(misconceptions)} misconceptions")
for m in misconceptions:
    print(f"  {m['id']}. {m['misconception']}")

Defined 5 misconceptions
  1. DataFrame.where() filters rows like SQL WHERE (Category 1.2 - Faulty Data Analysis Logic)
  2. Using != 0 and > 0 are equivalent for counting non-zero entries (Category 1.3 - Incorrect Data Handling)
  3. Using & vs 'and' on boolean pandas masks are interchangeable (Category 3.1 - Incorrect Programming Assumptions)
  4. Iterating over DataFrame rows with a for-loop is the normal correct approach (Category 2.1 - Suboptimal Coding)
  5. Replacing NaN with 0 before calculations is always safe and correct (Category 1.3 - Incorrect Data Handling)


In [3]:
TUTOR_SYSTEM_PROMPT = (
    "You are an expert Python tutor working with a beginner student.\n"
    "Your strict teaching rules are:\n"
    "1. NEVER give away the answer directly - always guide through questions\n"
    "2. Ask one focused question per response to probe the student's understanding\n"
    "3. Use the Socratic method - respond to wrong answers with 'interesting, what do you think would happen if...'\n"
    "4. Keep responses concise - 3-4 sentences maximum\n"
    "5. Only confirm the correct answer after the student has demonstrated understanding themselves\n"
    "Your goal is to identify the student's misconception within 4 turns and guide them to correct understanding by turn 8."
)

print("Tutor system prompt defined.")
print(f"Prompt length: {len(TUTOR_SYSTEM_PROMPT)} characters")

Tutor system prompt defined.
Prompt length: 620 characters


In [4]:
def call_with_backoff(messages, temperature=0.7, max_tokens=200, max_retries=5):
    """Call Groq API with exponential backoff on rate limit errors."""
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens
            )
            return response.choices[0].message.content
        except RateLimitError:
            wait = 2 ** attempt
            print(f"  Rate limit hit (attempt {attempt+1}/{max_retries}). Waiting {wait}s...")
            time.sleep(wait)
    raise RuntimeError("Max retries exceeded - still rate limited.")


def run_experiment(misconception, num_turns=6, verbose=True):
    student_history = [{"role": "system", "content": misconception["system_prompt"]}]
    tutor_history   = [{"role": "system", "content": TUTOR_SYSTEM_PROMPT}]
    transcript = []

    if verbose:
        print(f"\n{'='*60}")
        print(f"Experiment {misconception['id']}: {misconception['misconception']}")
        print(f"{'='*60}\n")

    student_message = misconception["opening"]

    for turn in range(num_turns):

        # Student speaks
        if verbose:
            print(f"STUDENT (turn {turn+1}): {student_message}\n")
        transcript.append({"turn": turn+1, "role": "student", "content": student_message})
        tutor_history.append({"role": "user", "content": student_message})

        # Tutor responds
        tutor_message = call_with_backoff(tutor_history, temperature=0.7, max_tokens=200)
        time.sleep(1)
        if verbose:
            print(f"TUTOR (turn {turn+1}): {tutor_message}\n")
        transcript.append({"turn": turn+1, "role": "tutor", "content": tutor_message})
        tutor_history.append({"role": "assistant", "content": tutor_message})

        # Student replies
        student_history.append({"role": "user",      "content": tutor_message})
        next_student_message = call_with_backoff(student_history, temperature=0.8, max_tokens=150)
        time.sleep(1)
        student_history.append({"role": "assistant", "content": next_student_message})
        student_message = next_student_message

    return transcript


print("Conversation loop defined successfully.")

Conversation loop defined successfully.


In [5]:
all_transcripts = []

for misconception in misconceptions:
    transcript = run_experiment(misconception, num_turns=6, verbose=True)
    all_transcripts.append({
        "misconception_id": misconception["id"],
        "misconception":    misconception["misconception"],
        "transcript":       transcript
    })
    print(f"\nExperiment {misconception['id']} complete - {len(transcript)} turns logged.\n")

print("All experiments complete.")


Experiment 1: DataFrame.where() filters rows like SQL WHERE (Category 1.2 - Faulty Data Analysis Logic)

STUDENT (turn 1): Hi, I'm trying to filter my DataFrame to only keep rows where age is over 18, but when I check the length after using .where() it's still the same as before. I wrote filtered = df.where(df['age'] > 18) - is my condition wrong?

TUTOR (turn 1): That's a good start with the condition. What do you think happens to the rows in your DataFrame where the condition `df['age'] > 18` is False when you use the `.where()` method?

STUDENT (turn 2): I think those rows should be removed from the DataFrame, so I'm expecting `len(filtered)` to be less than `len(df)`. But for some reason, the length of `filtered` is still the same as the original DataFrame.

TUTOR (turn 2): Interesting, what do you think would happen if those rows aren't exactly "removed", but instead are replaced with something else, like a specific value, when the condition is False?

STUDENT (turn 3): That's we

RuntimeError: Max retries exceeded - still rate limited.

In [ ]:
rubric_criteria = {
    "misconception_identified": {
        "description": "Did the tutor correctly identify the student's specific misconception?",
        "scale": "0 = not identified, 1 = partially identified, 2 = clearly identified"
    },
    "socratic_method": {
        "description": "Did the tutor guide through questions rather than stating answers directly?",
        "scale": "0 = gave answers directly, 1 = mixed, 2 = consistently Socratic"
    },
    "answer_withholding": {
        "description": "Did the tutor avoid revealing the solution (including in code snippets)?",
        "scale": "0 = gave away solution, 1 = partially withheld, 2 = fully withheld"
    },
    "student_understanding": {
        "description": "Did the student show signs of reaching correct understanding by the end?",
        "scale": "0 = no progress, 1 = partial progress, 2 = clear understanding reached"
    },
    "memory_retention": {
        "description": "Did the tutor consistently track and build on what the student already knew or didn't know?",
        "scale": "0 = ignored prior turns, 1 = occasionally referenced, 2 = consistently tracked student state"
    }
}

print(f"Rubric defined with {len(rubric_criteria)} criteria:")
for k in rubric_criteria:
    print(f"  - {k}")

In [ ]:
def evaluate_transcript(transcript, misconception, rubric_criteria, max_retries=3):

    transcript_text = ""
    for turn in transcript:
        transcript_text += f"{turn['role'].upper()} (turn {turn['turn']}): {turn['content']}\n\n"

    criteria_text = ""
    for criterion, details in rubric_criteria.items():
        criteria_text += f"\n{criterion}:\n"
        criteria_text += f"  Description: {details['description']}\n"
        criteria_text += f"  Scale: {details['scale']}\n"

    json_template = (
        '{"misconception_identified":{"score":0,"justification":"text"},'
        '"socratic_method":{"score":0,"justification":"text"},'
        '"answer_withholding":{"score":0,"justification":"text"},'
        '"student_understanding":{"score":0,"justification":"text"},'
        '"memory_retention":{"score":0,"justification":"text"}}'
    )

    evaluation_prompt = (
        f'Evaluate this Python tutoring conversation. The student\'s misconception is:\n'
        f'"{misconception}"\n\n'
        f'TRANSCRIPT:\n{transcript_text}\n\n'
        f'Score the tutor on each criterion (0, 1, or 2) with a one-sentence justification.\n\n'
        f'CRITERIA:{criteria_text}\n'
        f'IMPORTANT: Return ONLY a JSON object using this exact structure:\n'
        f'{json_template}'
    )

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {"role": "system", "content": "You are a JSON-only responder. Output valid JSON and nothing else."},
                    {"role": "user",   "content": evaluation_prompt}
                ],
                temperature=0.0,
                max_tokens=800
            )
            raw = response.choices[0].message.content.strip()
            if attempt == 0:
                print(f"  Raw preview: {raw[:100]}...")

            # Strip markdown fences if present
            if "```" in raw:
                for part in raw.split("```"):
                    if "{" in part:
                        raw = part
                        break
            if raw.startswith("json"):
                raw = raw[4:]
            start = raw.find("{")
            end   = raw.rfind("}") + 1
            if start != -1 and end > start:
                raw = raw[start:end]

            return json.loads(raw)

        except Exception as e:
            print(f"  Attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
            else:
                return {c: {"score": 0, "justification": "evaluation failed"} for c in rubric_criteria}


print("Running automated rubric evaluation...\n")

all_scores = []

for exp in all_transcripts:
    print(f"Evaluating experiment {exp['misconception_id']}: {exp['misconception']}...")
    scores_raw   = evaluate_transcript(exp['transcript'], exp['misconception'], rubric_criteria)
    total        = sum(v['score'] for v in scores_raw.values())
    max_possible = len(rubric_criteria) * 2

    result = {
        "experiment_id": exp['misconception_id'],
        "misconception":  exp['misconception'],
        "total":          total,
        "max_possible":   max_possible,
    }
    for criterion, data in scores_raw.items():
        result[f"{criterion}_score"]         = data['score']
        result[f"{criterion}_justification"] = data['justification']
    all_scores.append(result)

    print(f"\n  {'Criterion':<28} {'Score':<8} Justification")
    print(f"  {'-'*85}")
    for criterion, data in scores_raw.items():
        marker = " <-- KEY METRIC" if criterion == "memory_retention" else ""
        print(f"  {criterion:<28} {data['score']}/2    {data['justification']}{marker}")
    print(f"\n  Total: {total}/{max_possible}\n")

print("=" * 60)
print("Summary")
print("=" * 60)
print(f"{'Exp':<6} {'Misconception':<45} {'Score'}")
print("-" * 60)
for s in all_scores:
    print(f"{s['experiment_id']:<6} {s['misconception']:<45} {s['total']}/{s['max_possible']}")

In [ ]:
# Build fieldnames dynamically to match the actual keys in all_scores
base_fields     = ['experiment_id', 'misconception', 'total', 'max_possible']
criteria_fields = []
for criterion in rubric_criteria:
    criteria_fields.append(f"{criterion}_score")
    criteria_fields.append(f"{criterion}_justification")
fieldnames = base_fields + criteria_fields

with open('results.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for s in all_scores:
        writer.writerow({k: s[k] for k in fieldnames})

with open('transcripts.txt', 'w') as f:
    for exp in all_transcripts:
        f.write(f"{'='*60}\n")
        f.write(f"Experiment {exp['misconception_id']}: {exp['misconception']}\n")
        f.write(f"{'='*60}\n\n")
        for turn in exp['transcript']:
            f.write(f"{turn['role'].upper()} (turn {turn['turn']}): {turn['content']}\n\n")
        f.write("\n")

print("Saved: results.csv")
print("Saved: transcripts.txt")
print(f"CSV columns: {fieldnames}")